<a href="https://colab.research.google.com/github/bernardarthur0123-netizen/foundation-/blob/main/04_Transfer_Learning_ResNet18_PyTorch_EN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4 · Transfer Learning with ResNet18 in PyTorch

**Tema:** using a pretrained CNN and adapting it to a new problem

## 1. Idea central

Training a deep CNN from scratch may require lots of data and time. **Transfer learning** reuses a network that was already trained on a large dataset (for example, ImageNet) and adapts it to a new task.

In [ ]:

import copy
import os
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
torch.manual_seed(42)

## 2. Dataset: Hymenoptera (ants vs bees)

We will use a well-known public dataset from PyTorch tutorials: ants vs bees images. It is perfect to show how a general-purpose network can quickly specialize on a specific task.

In [ ]:

import urllib.request

data_dir = Path("./data/hymenoptera_data")
zip_path = Path("./data/hymenoptera_data.zip")

if not data_dir.exists():
    url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
    os.makedirs("./data", exist_ok=True)
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("./data")
print("Dataset path:", data_dir)

In [ ]:

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

image_datasets = {
    "train": datasets.ImageFolder(data_dir / "train", transform=train_transform),
    "val": datasets.ImageFolder(data_dir / "val", transform=val_transform),
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=16, shuffle=(x=="train"))
    for x in ["train", "val"]
}

dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "val"]}
class_names = image_datasets["train"].classes

dataset_sizes, class_names

In [ ]:

def imshow_tensor(inp, title=None):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    inp = (inp * std + mean).clamp(0, 1)
    plt.imshow(inp.permute(1, 2, 0))
    if title:
        plt.title(title)
    plt.axis("off")

inputs, classes = next(iter(dataloaders["train"]))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    plt.sca(ax)
    imshow_tensor(inputs[i], title=class_names[classes[i]])
plt.tight_layout()
plt.show()

## 3. Load a pretrained ResNet18

In [ ]:

from torchvision.models import ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(class_names))

model = model.to(device)
print(model)

Here we do **feature extraction**: we freeze the convolutional backbone and train only the last fully connected layer. This is very efficient when the new dataset is small.

In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

In [ ]:

def train_model(model, dataloaders, criterion, optimizer, device, num_epochs=5):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 40)

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    preds = outputs.argmax(dim=1)
                    loss = criterion(outputs, labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels).item()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects / dataset_sizes[phase]

            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)

            print(f"{phase} loss: {epoch_loss:.4f} acc: {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    model.load_state_dict(best_model_wts)
    return model, history

## 4. Training

In [ ]:

model, history = train_model(model, dataloaders, criterion, optimizer, device, num_epochs=5)

In [ ]:

epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(7,4))
plt.plot(epochs, history["train_loss"], label="train_loss")
plt.plot(epochs, history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss evolution")
plt.legend()
plt.show()

plt.figure(figsize=(7,4))
plt.plot(epochs, history["train_acc"], label="train_acc")
plt.plot(epochs, history["val_acc"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy evolution")
plt.legend()
plt.show()

## 5. Visualize predictions

In [ ]:

@torch.no_grad()
def visualize_predictions(model, dataloader, class_names, device, num_images=8):
    model.eval()
    images_shown = 0
    fig = plt.figure(figsize=(12, 6))

    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = outputs.argmax(dim=1)

        for j in range(inputs.size(0)):
            images_shown += 1
            ax = plt.subplot(2, 4, images_shown)
            ax.axis("off")
            ax.set_title(f"pred: {class_names[preds[j]]}")

            img = inputs[j].cpu()
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
            img = (img * std + mean).clamp(0,1)
            ax.imshow(img.permute(1,2,0))

            if images_shown == num_images:
                plt.tight_layout()
                return

In [ ]:

visualize_predictions(model, dataloaders["val"], class_names, device)
plt.show()

## 6. Two transfer learning strategies

### A. Feature extraction
- freeze the backbone
- train only the final head
- fast and effective with little data

### B. Fine-tuning
- unfreeze part or all of the network
- adapt pretrained weights to the new domain
- can perform better if you have more data

A typical example would be:

```python
for param in model.parameters():
    param.requires_grad = True
```

and then using a small learning rate.

---

## 7. Business example

Imagine an insurance company with only a few thousand labeled images of:
- minor damage
- medium damage
- severe damage

Training from scratch can be costly and unstable.
By contrast, reusing a CNN trained on millions of images lets you start from rich visual representations.

That is exactly the logic behind transfer learning.

---

## 8. Key ideas

- pretrained CNNs learn general-purpose useful representations
- transfer learning saves data, time, and compute
- ResNet introduces residual connections that help train deep networks
- freezing or fine-tuning depends on dataset size and domain similarity

---

## 9. Exercises

1. Replace `resnet18` with `efficientnet_b0`.  
2. Fine-tune the last residual block.  
3. Compute a validation confusion matrix.  
4. Try a 3-class or multi-class setup with `ImageFolder`.